In [1]:
"""
AUTORANK AI - REAL TF-IDF SKILL MATCHING SYSTEM
MSc Data Analytics Project
Dynamic model for job-candidate matching
"""

# ============================================
# IMPORT LIBRARIES
# ============================================
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import os

# ============================================
# CONFIGURATION - USING RELATIVE PATHS (PROFESSOR-PROOF)
# ============================================

# Get the current notebook's directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Since notebook is in 'notebooks' folder, go UP one level to find data
project_root = os.path.dirname(current_dir)  # This goes up one level
data_path = os.path.join(project_root, "data")
results_path = os.path.join(project_root, "results", "dynamic_results")

# File paths
candidates_file = os.path.join(data_path, "candidates.csv")
jobs_file = os.path.join(data_path, "jobs.csv")

# Create results directory if it doesn't exist
os.makedirs(results_path, exist_ok=True)

print("="*70)
print("AUTORANK AI - Dynamic Skill Matching System")
print("="*70)
print(f"Project root: {project_root}")
print(f"Data path: {data_path}")
print(f"Candidates: {candidates_file}")
print(f"Jobs: {jobs_file}")
print(f"Results will be saved to: {results_path}")
print("="*70)

# ============================================
# STEP 1: LOAD DATA
# ============================================
print("\n📂 STEP 1: Loading data...")

try:
    # Check if files exist before loading
    if not os.path.exists(candidates_file):
        print(f"❌ Candidates file not found at: {candidates_file}")
        print("\nPlease make sure your files are in the correct location:")
        print(f"  {data_path}")
        print("\nExpected files:")
        print("  - candidates.csv")
        print("  - jobs.csv")
        exit()
    
    if not os.path.exists(jobs_file):
        print(f"❌ Jobs file not found at: {jobs_file}")
        exit()
    
    candidates_df = pd.read_csv(candidates_file)
    jobs_df = pd.read_csv(jobs_file)
    print(f"✅ Loaded {len(candidates_df)} candidates")
    print(f"✅ Loaded {len(jobs_df)} jobs")
    
    # Display first few rows to verify
    print("\n📋 First 2 candidates:")
    print(candidates_df.head(2))
    print("\n📋 Jobs:")
    print(jobs_df)
    
except Exception as e:
    print(f"❌ Error loading files: {e}")
    exit()

# ============================================
# STEP 2: PREPARE SKILL TEXTS
# ============================================
print("\n🔧 STEP 2: Preparing skill texts...")

# Define skill columns (from your CSV)
skill_columns = ['Python', 'SQL', 'Tableau', 'Excel', 'SEO', 'Content', 
                 'GA', 'Django', 'AWS', 'Java', 'HTML', 'CSS', 'Social', 'PowerPoint']

def create_candidate_text(row):
    """Convert candidate skills to text for TF-IDF"""
    skills = []
    for skill in skill_columns:
        years = row.get(skill, 0)
        if years > 0:
            # Repeat skill name based on experience (more years = more weight)
            skills.extend([skill.lower()] * int(years))
    return ' '.join(skills)

def create_job_text(row):
    """Create job description text with required skills emphasized"""
    title = row.get('Title', '') if 'Title' in row else row.get('Job_Title', '')
    required = row.get('Required_Skills', '')
    
    # Create enhanced text with skills repeated for weighting
    text = f"{title} {required}"
    # Add required skills multiple times to give them more weight
    if pd.notna(required):
        skills = str(required).split(',')
        for skill in skills:
            skill = skill.strip().lower()
            text += f" {skill} " * 3  # Repeat 3 times for emphasis
    
    return text.lower()

# Create text representations
candidates_df['skill_text'] = candidates_df.apply(create_candidate_text, axis=1)
jobs_df['job_text'] = jobs_df.apply(create_job_text, axis=1)

print("✅ Created text representations")
print("\nSample candidate text (first 200 chars):")
print(candidates_df['skill_text'].iloc[0][:200], "...")
print("\nSample job text (first 200 chars):")
print(jobs_df['job_text'].iloc[0][:200], "...")

# ============================================
# STEP 3: TF-IDF VECTORIZATION
# ============================================
print("\n📊 STEP 3: Computing TF-IDF vectors...")

# Combine all texts for vectorization
all_texts = pd.concat([candidates_df['skill_text'], jobs_df['job_text']])

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),  # Use unigrams and bigrams
    max_features=100,      # Limit to top 100 features
    min_df=1
)

# Fit and transform
tfidf_matrix = vectorizer.fit_transform(all_texts)

# Split back into candidates and jobs
n_candidates = len(candidates_df)
candidate_vectors = tfidf_matrix[:n_candidates]
job_vectors = tfidf_matrix[n_candidates:]

print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"   - {n_candidates} candidate vectors")
print(f"   - {len(jobs_df)} job vectors")

# Show top features
feature_names = vectorizer.get_feature_names_out()
print("\nTop 20 features extracted:")
for i, feature in enumerate(feature_names[:20], 1):
    print(f"  {i}. {feature}")

# ============================================
# STEP 4: COMPUTE SIMILARITY SCORES
# ============================================
print("\n📈 STEP 4: Computing similarity scores...")

# Compute cosine similarity between each job and all candidates
similarity_matrix = cosine_similarity(job_vectors, candidate_vectors)

print(f"✅ Similarity matrix shape: {similarity_matrix.shape}")
print(f"   ({len(jobs_df)} jobs × {len(candidates_df)} candidates)")

# ============================================
# STEP 5: INCORPORATE EXPERIENCE WEIGHTING
# ============================================
print("\n⚖️ STEP 5: Applying experience weighting...")

def calculate_experience_score(candidate_row, job_skills):
    """Calculate experience match score"""
    if not job_skills or pd.isna(job_skills):
        return 0
    
    # Parse job skills
    required_skills = [s.strip().lower() for s in str(job_skills).split(',')]
    
    # Get candidate's years for matching skills
    total_years = 0
    matched_skills = 0
    
    for skill in required_skills:
        # Find matching skill column (case-insensitive)
        for col in skill_columns:
            if col.lower() == skill or skill in col.lower():
                years = candidate_row.get(col, 0)
                if years > 0:
                    total_years += years
                    matched_skills += 1
                break
    
    if matched_skills == 0:
        return 0
    
    # Average years for matched skills
    avg_years = total_years / matched_skills
    
    # Assume 3 years expected experience
    expected_years = 3
    exp_score = min(avg_years / expected_years, 1.0)
    
    return exp_score

# Create results for each job
all_results = {}

for job_idx, job_row in jobs_df.iterrows():
    job_title = job_row.get('Title', job_row.get('Job_Title', f'Job_{job_idx}'))
    job_skills = job_row.get('Required_Skills', '')
    
    print(f"\n📋 Processing: {job_title}")
    
    # Get base similarity scores
    sim_scores = similarity_matrix[job_idx]
    
    # Calculate final scores with experience weighting
    results = []
    for cand_idx, cand_row in candidates_df.iterrows():
        base_score = sim_scores[cand_idx]
        exp_score = calculate_experience_score(cand_row, job_skills)
        
        # Final score: 70% skill similarity, 30% experience
        final_score = (0.7 * base_score) + (0.3 * exp_score)
        
        # Get candidate name
        cand_name = cand_row.get('Name', f'Candidate_{cand_idx}')
        
        # Determine tier
        if final_score >= 0.8:
            tier = "🔵 HIGH"
        elif final_score >= 0.5:
            tier = "🟢 MEDIUM"
        else:
            tier = "🟡 LOW"
        
        results.append({
            'Candidate_ID': cand_row.get('ID', f'C{cand_idx+1}'),
            'Name': cand_name,
            'Base_Similarity': round(base_score, 3),
            'Experience_Score': round(exp_score, 3),
            'Final_Score': round(final_score, 3),
            'Final_Percentage': f"{round(final_score * 100, 1)}%",
            'Tier': tier
        })
    
    # Sort by final score (descending)
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    
    # Add ranking (competition style)
    results_df['Rank'] = results_df['Final_Score'].rank(method='min', ascending=False).astype(int)
    
    # Store results
    all_results[job_title] = results_df
    
    # Display top results
    print(f"\nTop candidates for {job_title}:")
    display_cols = ['Rank', 'Name', 'Final_Percentage', 'Tier', 'Base_Similarity', 'Experience_Score']
    print(results_df[display_cols].head(8).to_string(index=False))
    print("-"*70)

# ============================================
# STEP 6: SAVE RESULTS
# ============================================
print("\n💾 STEP 6: Saving results...")

for job_title, results_df in all_results.items():
    # Create safe filename
    safe_title = re.sub(r'[^\w\s-]', '', job_title).strip().replace(' ', '_')
    filename = os.path.join(results_path, f"{safe_title}_rankings.csv")
    results_df.to_csv(filename, index=False)
    print(f"✅ Saved: {filename}")

# Also save summary file
summary_data = []
for job_title, results_df in all_results.items():
    top_candidate = results_df.iloc[0]['Name']
    top_score = results_df.iloc[0]['Final_Percentage']
    tier_counts = results_df['Tier'].value_counts()
    
    summary_data.append({
        'Job': job_title,
        'Top_Candidate': top_candidate,
        'Top_Score': top_score,
        'High_Count': tier_counts.get('🔵 HIGH', 0),
        'Medium_Count': tier_counts.get('🟢 MEDIUM', 0),
        'Low_Count': tier_counts.get('🟡 LOW', 0),
        'Total_Candidates': len(results_df)
    })

summary_df = pd.DataFrame(summary_data)
summary_file = os.path.join(results_path, "summary_report.csv")
summary_df.to_csv(summary_file, index=False)
print(f"✅ Saved summary: {summary_file}")

# ============================================
# STEP 7: SUMMARY
# ============================================
print("\n" + "="*70)
print("✅ PROCESS COMPLETE!")
print("="*70)
print(f"\nResults saved to: {results_path}")
print("\nFiles created:")
if os.path.exists(results_path):
    for f in os.listdir(results_path):
        print(f"   - {f}")
else:
    print("   No files found yet")
print("\n" + "="*70)

Current directory: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\notebooks
AUTORANK AI - Dynamic Skill Matching System
Project root: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI
Data path: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\data
Candidates: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\data\candidates.csv
Jobs: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\data\jobs.csv
Results will be saved to: C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\results\dynamic_results

📂 STEP 1: Loading data...
✅ Loaded 8 candidates
✅ Loaded 3 jobs

📋 First 2 candidates:
   ID        Name  Python  SQL  Tableau  Excel  SEO  Content  GA  Django  AWS  \
0  C1  Alice Chen       4    4        3      5    0        0   0       0    0   
1  C2   Bob Smith       1    0        0      4    0        0   0       0    0   

   Java  HTML  CSS  Social  PowerPoint         Current_Role  
0     0     0    0       0           0  Senior Data Analyst  
1  

In [ ]:
# ============================================
# GRAPH GENERATION FOR AUTORANK AI RESULTS
# ============================================

import matplotlib.pyplot as plt
import pandas as pd
import os

# Set paths
results_path = r"C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\results\dynamic_results"
graphs_path = r"C:\Users\LENOVO\Desktop\HKBU\sem 2\projects\AutoRank-AI\graphs"

# Create graphs folder if it doesn't exist
os.makedirs(graphs_path, exist_ok=True)

# Load the results
da_df = pd.read_csv(os.path.join(results_path, "Data_Analyst_rankings.csv"))
mm_df = pd.read_csv(os.path.join(results_path, "Marketing_Manager_rankings.csv"))
se_df = pd.read_csv(os.path.join(results_path, "Software_Engineer_rankings.csv"))

print("="*60)
print("📊 GENERATING GRAPHS FOR GITHUB")
print("="*60)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# ============================================
# GRAPH 1: Tier Distribution - Data Analyst
# ============================================
print("\n📈 Generating Graph 1: Tier Distribution...")

tier_counts = da_df['Tier'].value_counts()
tiers = ['🔵 HIGH', '🟢 MEDIUM', '🟡 LOW']
counts = [
    tier_counts.get('🔵 HIGH', 0),
    tier_counts.get('🟢 MEDIUM', 0),
    tier_counts.get('🟡 LOW', 0)
]
colors = ['#1f77b4', '#2ca02c', '#ff7f0e']

plt.figure(figsize=(8, 6))
bars = plt.bar(tiers, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{int(height)}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.title('Data Analyst - Candidate Tier Distribution', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Number of Candidates', fontsize=14)
plt.ylim(0, 5)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graphs_path, 'tier_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(graphs_path, 'tier_distribution.png')}")

# ============================================
# GRAPH 2: Top Candidates - Data Analyst
# ============================================
print("\n📈 Generating Graph 2: Top Candidates...")

top5 = da_df.head(5)
candidates = top5['Name'].tolist()
scores = [float(s.replace('%', '')) for s in top5['Final_Percentage']]

# Color based on score
colors = []
for score in scores:
    if score >= 80:
        colors.append('#1f77b4')  # Blue
    elif score >= 50:
        colors.append('#2ca02c')  # Green
    else:
        colors.append('#ff7f0e')  # Orange

plt.figure(figsize=(12, 6))
bars = plt.barh(candidates, scores, color=colors, alpha=0.8, edgecolor='black', linewidth=1)

for bar, score in zip(bars, scores):
    plt.text(score + 1, bar.get_y() + bar.get_height()/2,
             f'{score}%', va='center', fontsize=12, fontweight='bold')

plt.title('Data Analyst - Top 5 Candidates', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Match Score (%)', fontsize=14)
plt.xlim(0, 100)
plt.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1f77b4', label='🔵 HIGH (80-100%)'),
    Patch(facecolor='#2ca02c', label='🟢 MEDIUM (50-79%)'),
    Patch(facecolor='#ff7f0e', label='🟡 LOW (<50%)')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(graphs_path, 'top_candidates.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(graphs_path, 'top_candidates.png')}")

# ============================================
# GRAPH 3: Skill Gap Analysis
# ============================================
print("\n📈 Generating Graph 3: Skill Gap Analysis...")

# From your results
skills = ['Tableau', 'Django', 'Google Analytics', 'AWS', 'SQL']
missing = [6, 5, 7, 4, 2]

plt.figure(figsize=(10, 6))
bars = plt.bar(skills, missing, color='#d62728', alpha=0.8, edgecolor='black', linewidth=1.5)

for bar, count in zip(bars, missing):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.2,
             f'{count}/8', ha='center', va='bottom', fontsize=12, fontweight='bold')
    plt.text(bar.get_x() + bar.get_width()/2., height/2,
             f'{(count/8)*100:.0f}%', ha='center', va='center', color='white', fontweight='bold')

plt.title('Most Common Missing Skills Across All Jobs', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Candidates Missing (out of 8)', fontsize=14)
plt.ylim(0, 8)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graphs_path, 'skill_gaps.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(graphs_path, 'skill_gaps.png')}")

# ============================================
# GRAPH 4: Alice Chen - Cross Job Comparison
# ============================================
print("\n📈 Generating Graph 4: Alice Chen Cross-Job Comparison...")

# Get Alice's scores
alice_da = da_df[da_df['Name'] == 'Alice Chen']['Final_Percentage'].values[0]
alice_mm = mm_df[mm_df['Name'] == 'Alice Chen']['Final_Percentage'].values[0]
alice_se = se_df[se_df['Name'] == 'Alice Chen']['Final_Percentage'].values[0]

alice_scores = [
    float(alice_da.replace('%', '')),
    float(alice_mm.replace('%', '')),
    float(alice_se.replace('%', ''))
]

jobs = ['Data Analyst', 'Marketing Manager', 'Software Engineer']
colors = ['#2ca02c' if s >= 50 else '#ff7f0e' for s in alice_scores]

plt.figure(figsize=(10, 6))
bars = plt.bar(jobs, alice_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

for bar, score in zip(bars, alice_scores):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{score}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.title('Alice Chen - Performance Across Different Jobs', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Match Score (%)', fontsize=14)
plt.ylim(0, 80)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graphs_path, 'alice_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(graphs_path, 'alice_comparison.png')}")

# ============================================
# GRAPH 5: Manual vs TF-IDF Comparison
# ============================================
print("\n📈 Generating Graph 5: Manual vs TF-IDF Comparison...")

candidates_comp = ['Alice', 'Charlie', 'Fiona', 'Helen', 'George', 'Bob']
manual_scores = [100, 100, 77, 77, 43, 33]
tfidf_scores = [73.2, 89.5, 51.4, 59.9, 22.9, 11.6]

x = range(len(candidates_comp))
width = 0.35

plt.figure(figsize=(14, 7))
bars1 = plt.bar([i - width/2 for i in x], manual_scores, width, label='Manual Baseline', color='#1f77b4', alpha=0.8)
bars2 = plt.bar([i + width/2 for i in x], tfidf_scores, width, label='TF-IDF Dynamic', color='#ff7f0e', alpha=0.8)

for bar, score in zip(bars1, manual_scores):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{score}%', ha='center', va='bottom', fontsize=10)

for bar, score in zip(bars2, tfidf_scores):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{score}%', ha='center', va='bottom', fontsize=10)

plt.title('Data Analyst: Manual Baseline vs TF-IDF Model', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Candidates', fontsize=14)
plt.ylabel('Match Score (%)', fontsize=14)
plt.xticks(x, candidates_comp)
plt.ylim(0, 110)
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(graphs_path, 'comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(graphs_path, 'comparison.png')}")

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*60)
print("✅ ALL GRAPHS GENERATED SUCCESSFULLY!")
print("="*60)
print(f"\nGraphs saved to: {graphs_path}")
print("\nFiles created:")
for f in os.listdir(graphs_path):
    print(f"   - {f}")
print("="*60)